In [ ]:
from darts import TimeSeries
from darts.metrics import ae, mae, rmse
from darts.utils.timeseries_generation import sine_timeseries, linear_timeseries
from datetime import datetime, timedelta, UTC
import pandas as pd
import numpy as np

from darts.metrics.utils import (
    _get_values_or_raise,
)
import pytz

Very dirty playground notebook used to develop and test the custom daily peak metric. I know this is something where unit tests would be great but have you considered I'm lazy rn?

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
tz = pytz.timezone("Europe/Zurich")

In [ ]:
now = datetime.now(tz)
today = datetime(now.year, now.month, now.day, tzinfo=now.tzinfo)
print(f"Today in {tz}: {today}")
today_utc = today.astimezone(UTC)
print(f"Today in UTC: {today_utc}")
start = pd.Timestamp(today).tz_convert(None)  # remove time zone because darts can't handle that
# end seems to be inclusive (97 hours), we want exclusive (96 hours)
end = start + timedelta(days=4) - timedelta(seconds=1)
# and for testing we want a partial day in the end
end -= timedelta(hours=5)
peak = 1.0


def _get_sine(peak: float, phase_shift=-3 * np.pi / 4):
    # value_phase = peak just before noon
    return sine_timeseries(
        value_phase=phase_shift,
        value_frequency=1 / 24,
        value_y_offset=peak / 2,
        value_amplitude=peak / 2,
        start=start,
        end=end,
        freq="1h",
    )


base = _get_sine(1.0)
trend = linear_timeseries(0, 0.5, start, end, freq="1h")
# trend = 0
actual = base + trend
actual = actual.with_columns_renamed("sine", "actual")
# take out a chunk to simulate missing data
values = actual.values()
values[60:66] = np.nan
# values[50:56] = np.nan
actual = actual.with_values(values)

In [ ]:
pred = _get_sine(1.5, phase_shift=-2 * np.pi / 4) + trend * 2
pred = pred.with_columns_renamed("sine", "pred")

In [ ]:
# REMEMBER: the plots show UTC time, like the underlying data! Could change with rcParams['timezone']
actual.plot()
pred.plot()

In [ ]:
mae(actual, pred), rmse(actual, pred)

In [ ]:
def peak_diffs(actual: TimeSeries, pred: TimeSeries):
    assert isinstance(actual, TimeSeries)
    assert isinstance(pred, TimeSeries)

    print("Shape in actual:", actual.shape)
    print("Shape in pred:", pred.shape)
    # assert isinstance(actual.time_index, pd.DatetimeIndex)
    # t = actual.time_index.to_series().dt.date.to_numpy() <- same as below
    # t = actual.time_index.date
    # add the day as a component to group by when doing daily calculations on numpy array
    # typing bug in darts: https://github.com/unit8co/darts/issues/2926
    actual = actual.add_datetime_attribute("day", tz=tz)  # pyright: ignore[reportArgumentType]
    pred = pred.add_datetime_attribute("day", tz=tz)  # pyright: ignore[reportArgumentType]
    # day_index = actual.columns.to_list().index("day")
    # print(day_index)

    y_true, y_pred = _get_values_or_raise(actual, pred, intersect=True, remove_nan_union=False)
    print(y_true.shape, y_pred.shape)

    # day is added, so use -1 to refer to the last component
    # print(np.unique(y_true[:, -1, :], return_index=True))
    # could also use pandas but I imagine it's slower, despite loop (not tested smile)
    def _get_max(arr: np.ndarray):
        # https://stackoverflow.com/a/43094244
        days = np.split(arr[:, :-1, :], np.unique(arr[:, -1, :], return_index=True)[1][1:])
        print(len(days))
        max_days = []
        for day in days:
            print(day.shape)
            # max over time (= per component and sample)
            m = np.nanmax(day, axis=0)
            print(m, f"({m.shape})")
            max_days.append(np.full_like(day, m))
        return np.concat(max_days)

    max_true = _get_max(y_true)
    # print(max_true)

    max_pred = _get_max(y_pred)
    # print(max_pred)

    return max_pred - max_true


peak_diffs(actual, pred).shape

In [ ]:
peak_diffs(actual, pred)

In [ ]:
peak_diffs(actual.add_datetime_attribute("hour"), pred.add_datetime_attribute("hour"))

In [ ]:
ae(actual, pred)

In [ ]:
def custom_ae(actual_series: TimeSeries, pred_series: TimeSeries, intersect=True, q=None):
    y_true, y_pred = _get_values_or_raise(
        actual_series,
        pred_series,
        intersect,
        remove_nan_union=False,
        q=q,
    )
    return np.abs(y_true - y_pred)


# when re-implementing the ae, it actually returns a different shape,
# so those decorates are definitely doing something even in the univariate case
custom_ae(actual, pred)

In [ ]:
from aare.evaluation.custom_metrics.daily_peak import adpd

m = adpd(actual, pred, tz=tz)
print(m)
np.mean(m)

I did some manual testing by varying the peak, the trend, the trend multiplier, the gap location and more. Seems to work :)

In [ ]:
# even multivariate seems to work :)
adpd(actual.add_datetime_attribute("hour"), pred.add_datetime_attribute("hour"), tz=tz)

What I didn't test yet is multiple samples i.e. stochastic series.

Done now below and implemented in module directly.

In [ ]:
pred_prob = pred.concatenate(pred + 0.5, axis="sample").concatenate(pred - 0.5, axis="sample")
pred_prob

In [ ]:
adpd(actual, pred_prob, tz=tz)

In [ ]:
ae(actual, pred_prob)

In [ ]:
from datetime import tzinfo
import darts


MONTH = pd.Timedelta(days=28)


def add_day_attribute(ts: TimeSeries, tz: str | tzinfo | None = None) -> TimeSeries:
    attribute = "day"
    assert isinstance(ts.duration, pd.Timedelta)
    if ts.duration >= MONTH:
        raise ValueError("Metric currently only supports slices shorter than a month because I was lazy.")

    if ts.is_deterministic:
        return ts.add_datetime_attribute(attribute, tz=tz)  # pyright: ignore[reportArgumentType]

    ts_uni = ts.with_values(ts.values(sample=0))  # take first sample
    # add_datetime_attribute works now as its univariate
    with_date = ts_uni.add_datetime_attribute(attribute, tz=tz)  # pyright: ignore[reportArgumentType]
    # must create a ts that matches in sample dimension to add as a component
    day_vals = with_date[attribute]
    day_full = darts.concatenate([day_vals] * ts.n_samples, axis="sample", ignore_time_axis=True)

    return ts.concatenate(day_full, axis="component")


add_day_attribute(pred_prob)

In [ ]:
from aare.evaluation.custom_metrics.daily_peak import adpd, madpd, rmsdpd

pd.DataFrame(
    [
        dict(
            mae=mae(actual, pred),
            rmse=rmse(actual, pred),
            madpd=madpd(actual, pred, tz=tz),
            rmsdpd=rmsdpd(actual, pred, tz=tz),
        )
    ]
).T